## Q1. What is Tableau? Explain its importance in Business Intelligence and how it helps in data-driven decision-making.

Tableau is a powerful, interactive data visualization and business intelligence tool that allows users to connect to various data sources and create dashboards, reports, and charts without requiring programming knowledge.

**Importance in Business Intelligence:**
- Enables non-technical users to explore data visually and uncover insights quickly.
- Supports real-time data connectivity with databases, cloud services, spreadsheets, and APIs.
- Allows drag-and-drop creation of complex visualizations such as maps, heat maps, scatter plots, and tree maps.
- Facilitates data-driven decisions by presenting KPIs, trends, and comparisons in an interactive format that stakeholders can filter and drill into.

**How it helps in decision-making:**
Tableau reduces the time from raw data to insight by allowing business users to see patterns, outliers, and trends instantly. For example, a sales manager can identify underperforming regions by interacting with a map visualization rather than reading rows in a spreadsheet.

## Q2. Explain the role of the following Tableau components:

**a) Data Pane:**
The Data Pane appears on the left side of Tableau Desktop and lists all fields from the connected data source, organized into Dimensions (qualitative) and Measures (quantitative). Users drag fields from the Data Pane to build views.

**b) Worksheet:**
A Worksheet is a single canvas where a visualization is built. Each workbook can contain multiple worksheets. You can place charts, tables, and maps on a worksheet.

**c) Dashboard:**
A Dashboard is a collection of multiple worksheets and objects arranged on a single view. It provides a consolidated view of multiple charts and allows interactive filtering using actions.

**d) Story:**
A Story in Tableau is a sequence of dashboards or worksheets arranged in a narrative order. It presents insights step-by-step, guiding the viewer through the data like a presentation slide deck.

## Q3. What is the difference between Dimensions and Measures in Tableau? Provide examples of each.

| Feature | Dimensions | Measures |
|---|---|---|
| Type | Qualitative / Categorical | Quantitative / Numerical |
| Role | Used for grouping, slicing, and filtering | Used for calculations and aggregations |
| Default aggregation | None | SUM, AVG, COUNT, etc. |
| Displayed as | Headers / Labels | Axis values |

**Examples of Dimensions:** Country, Product Category, Customer Name, Order Date (when used as a label)

**Examples of Measures:** Sales, Profit, Quantity, Discount, Order ID (when used for counting)

In Tableau, Dimensions are shown in blue and Measures in green in the Data Pane. When a Measure is dragged to the view, Tableau automatically aggregates it.

## Q4. Define and explain the purpose of Filters, Parameters, and Sets in Tableau.

**Filters:**
Filters restrict the data displayed in a view. There are several types:
- *Extract Filters*: Applied when extracting data from the source.
- *Data Source Filters*: Applied before any other filter, at the connection level.
- *Context Filters*: Create an independent filter context for other filters to work within.
- *Dimension / Measure Filters*: Applied to specific fields in the view.

**Parameters:**
A Parameter is a dynamic placeholder that stores a user-defined value (a number, string, or date). It allows users to input values that control calculations, reference lines, bin sizes, or filter conditions. Example: A parameter called 'Top N' can let users choose to see the top 5, 10, or 20 items.

**Sets:**
A Set is a custom field that defines a subset of data members based on conditions. Sets can be fixed (manually selected members) or dynamic (condition-based). Example: A set of 'High Value Customers' who spent over a threshold can be used to compare IN vs OUT group behaviors.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv('sample_superstore.csv')
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Sales'] = df['Quantity'] * 10  # Synthetic sales proxy
df.head()

## Q5. Gross Sales by Country — Bar Chart

In [ ]:
sales_country = df.groupby('Country')['Sales'].sum().reset_index()
fig = px.bar(sales_country, x='Country', y='Sales', title='Gross Sales by Country')
fig.show()

## Q6. Dual-axis Sales (bar) vs Profit (line) for 2014

In [ ]:
df_2014 = df[df['Order Date'].dt.year == 2014]
monthly = df_2014.groupby(df_2014['Order Date'].dt.month).agg({'Sales': 'sum', 'Profit': 'sum'})
fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Bar(x=monthly.index, y=monthly['Sales'], name='Sales'), secondary_y=False)
fig.add_trace(go.Scatter(x=monthly.index, y=monthly['Profit'], name='Profit'), secondary_y=True)
fig.update_layout(title='Monthly Sales vs Profit (2014)')
fig.show()

## Q7. Filled Map — Units Sold by Country

In [ ]:
units_country = df.groupby('Country')['Quantity'].sum().reset_index()
fig = px.choropleth(
    units_country,
    locations='Country',
    locationmode='country names',
    color='Quantity',
    title='Units Sold by Country'
)
fig.show()

## Q8. KPI Tiles + Profit Trend + Filters Simulation

In [ ]:
print('Total Profit KPI:', df['Profit'].sum())
trend = df.groupby(df['Order Date'].dt.to_period('M'))['Profit'].sum().reset_index()
trend['Order Date'] = trend['Order Date'].astype(str)
fig = px.line(trend, x='Order Date', y='Profit', title='Profit Trend Over Time')
fig.show()

## Q9. Low-Profit, High-Sales Products (Scatter Analysis)

In [ ]:
prod = df.groupby('Product Name')[['Quantity', 'Profit']].sum().reset_index()
fig = px.scatter(
    prod, x='Quantity', y='Profit',
    hover_name='Product Name',
    title='Volume vs Profit per Product'
)
fig.add_hline(y=0, line_dash='dash', line_color='red')
fig.show()
# Low profit / high volume = bottom-right quadrant: review pricing strategy for these products.

## Q10. Online Retail — Customer Retention Strategy

In [ ]:
retail = pd.read_csv('online_retail.csv')
retail['InvoiceDate'] = pd.to_datetime(retail['InvoiceDate'])
retail['Revenue'] = retail['Quantity'] * retail['UnitPrice']

repeats = retail.groupby('CustomerID')['InvoiceNo'].nunique() > 1
print('Repeat customers:', repeats.sum())

top_rev = retail.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(10).reset_index()
fig = px.bar(top_rev, x='Country', y='Revenue', title='Top 10 Countries by Revenue')
fig.show()

# Retention strategy: Offer loyalty discounts to repeat customers in top-revenue countries.
# Use RFM segmentation (Recency, Frequency, Monetary) to identify at-risk customers.